# Train **atlas** on Google Colab

Unsupervised contrastive training (ResNet-50) for ImageNette, then the few-label evaluation and figures — end to end.

**1. Pick your GPU:** `Runtime → Change runtime type →`
- **A100** or **L4** if your Colab Pro unlocks them — fastest (~15–30 min), best result.
- **T4** (free) also works (~1.5–2 h); lower the batch (see cell 4).

**2.** `Runtime → Run all`. Device + mixed-precision (AMP) are automatic — nothing to configure.

The encoder checkpoints every 25 epochs (`models/imagenette_encoder.pt`), so a preemption won't lose everything. The last cell downloads the artifacts.

In [ ]:
# 1) confirm the GPU
!nvidia-smi -L
import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU — set Runtime→GPU')

In [ ]:
# 2) clone the repo + install the light deps (torch is already on Colab)
!git clone -q https://github.com/cleoanka/atlas.git
%cd atlas
!pip install -q numpy scipy scikit-learn matplotlib pillow

In [ ]:
# 3) fetch ImageNette (~95 MB)
!mkdir -p data && curl -sL https://s3.amazonaws.com/fast-ai-imageclas/imagenette2-160.tgz -o data/imagenette2-160.tgz
!tar -xzf data/imagenette2-160.tgz -C data/ && echo 'ready:' && ls data/imagenette2-160

In [ ]:
# 4) TRAIN (unsupervised, ResNet-50). AMP auto-on on CUDA; checkpoints every 25 epochs.
#    Batch per GPU (VRAM):  A100 40GB -> 512 (and --epochs 800)  |  L4 24GB -> 384  |  T4 16GB -> 256
!python src/train_contrastive_imagenette.py --backbone resnet50 --img 160 --batch 384 --epochs 400

In [ ]:
# 5) evaluate + figures with the freshly trained encoder
!python src/evaluate.py --dataset imagenette
!python src/sweep.py    --dataset imagenette
!python figures/make_imagenette_figures.py

In [ ]:
# 6) bundle everything into one zip and download it (Colab is ephemeral)
!zip -q -r atlas_out.zip data/imagenette_emb.npz data/imagenette_train_emb.npz \
    models/imagenette_encoder.pt results/imagenette_results.json results/imagenette_sweep.json \
    figures/imagenette_*.png 2>/dev/null; ls -lh atlas_out.zip
from google.colab import files
files.download('atlas_out.zip')

### Keep it (optional)

- **Google Drive** instead of downloading: `from google.colab import drive; drive.mount('/content/drive')` then `!cp data/imagenette_emb.npz models/imagenette_encoder.pt /content/drive/MyDrive/`.
- **Push back to the repo:** set a token, `!git config user.email you@x` / `user.name you`, then commit the new `data/imagenette_emb.npz` + `results/*.json` + `figures/imagenette_*.png` and `git push`.
- The `.pt` encoder is what you keep to embed new images later; the `*_emb.npz` files are what the repo's evaluation/figures use.